# Data Exploration
This notebook explores the images in `data/interim/` for the TB detection project.
It demonstrates basic statistics, sample visualisation, intensity histograms and HOG features.


In [1]:
import os, glob, random
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from skimage.feature import hog
from skimage import exposure

# Root directory for interim data
DATA_ROOT = os.path.abspath('data/interim')

# Helper to collect image paths and class label (normal/TB) based on folder name
def collect_images(root):
    images = []
    # Accept folder names containing 'normal' or 'TB' case‑insensitively
    for class_dir in ['normal', 'TB']:
        pattern = os.path.join(root, '**', f'*{class_dir}*', '*.png')
        for path in glob.glob(pattern, recursive=True):
            # Determine label from folder name (case‑insensitive)
            label = 'normal' if 'normal' in path.lower() else 'TB'
            images.append({'path': path, 'label': label})
    return images

imgs = collect_images(DATA_ROOT)
print(f'Total images found: {len(imgs)}')

# Basic stats DataFrame
df = pd.DataFrame(imgs)
def img_stats(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    h, w = img.shape
    return h, w, img.mean()
df[['height','width','mean_intensity']] = df['path'].apply(lambda p: pd.Series(img_stats(p)))
print(df.groupby('label').agg({'height':'mean','width':'mean','mean_intensity':'mean'}))
print('\nClass distribution:')
print(df['label'].value_counts())


ModuleNotFoundError: No module named 'cv2'

In [ ]:
# Visual grid of random samples per class
def show_grid(df, n=5):
    fig, axes = plt.subplots(2, n, figsize=(n*3,6))
    for i, label in enumerate(['normal','TB']):
        subset = df[df['label']==label].sample(n, random_state=42)
        for ax, row in zip(axes[i], subset.itertuples()):
            img = cv2.imread(row.path, cv2.IMREAD_GRAYSCALE)
            ax.imshow(img, cmap='gray')
            ax.set_title(label)
            ax.axis('off')
    plt.tight_layout()
    plt.show()

show_grid(df)


In [ ]:
# Intensity histogram per class
def plot_histograms(df):
    plt.figure(figsize=(8,4))
    for label, color in zip(['normal','TB'], ['blue','red']):
        intensities = []
        for path in df[df['label']==label]['path'][:200]:  # limit for speed
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            intensities.extend(img.ravel())
        plt.hist(intensities, bins=256, alpha=0.5, color=color, label=label, density=True)
    plt.xlabel('Pixel intensity')
    plt.ylabel('Density')
    plt.legend()
    plt.title('Pixel intensity distribution')
    plt.show()

plot_histograms(df)


In [ ]:
# HOG visualisation for a few examples
def show_hog_example(df, label):
    sample = df[df['label']==label].sample(1, random_state=42).iloc[0]
    img = cv2.imread(sample.path, cv2.IMREAD_GRAYSCALE)
    fd, hog_image = hog(img, orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2), visualize=True, channel_axis=None)
    hog_image = exposure.rescale_intensity(hog_image, in_range=(0, 0.02))
    fig, ax = plt.subplots(1,2, figsize=(8,4))
    ax[0].imshow(img, cmap='gray')
    ax[0].set_title(f'{label} – original')
    ax[0].axis('off')
    ax[1].imshow(hog_image, cmap='gray')
    ax[1].set_title('HOG')
    ax[1].axis('off')
    plt.show()

show_hog_example(df, 'normal')
show_hog_example(df, 'TB')
